# Esperimento composito 03 — separabilità della coordinata 8

Test al buio: partiamo dal composito vincente ConvLSTM + tre GRU locali e verifichiamo se la coordinata di stato 8 possiede una transizione condizionalmente locale. Il candidato aggiunge una quarta `torch.nn.GRU`; il controllo aggiunge la stessa capacità alla testa globale senza separare la coordinata.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile
def project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    work = Path('/kaggle/working')
    if work.exists(): candidates += [p.parent for p in work.glob('*/pyproject.toml')]
    for candidate in candidates:
        marker = candidate / 'pyproject.toml'
        if marker.exists() and 'hay-single-compartment' in marker.read_text(): return candidate
    destination = Path(tempfile.mkdtemp(prefix='hay_composite_03_', dir='/kaggle/working'))
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(destination)])
    return destination
ROOT = project_root(); SRC = ROOT / 'src'
assert (SRC / 'hay_single_compartment').is_dir()
sys.path.insert(0, str(SRC)); print('Project:', ROOT)

In [ ]:
import h5py, json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hay_single_compartment import INPUT_NAMES, STATE_NAMES, SimulationConfig, generate_dataset, validate_dataset
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_batch, train_model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/hay_composite_experiment_03') if Path('/kaggle').exists() else ROOT / 'artifacts' / 'composite_03'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET = OUTPUT_DIR / 'single_compartment_composite_v1.h5'
print('Device:', DEVICE, '| output:', OUTPUT_DIR)

In [ ]:
config = SimulationConfig(duration_ms=500.0, warmup_ms=150.0, seed=27182, train_trajectories=24, validation_trajectories=4, test_trajectories=6)
dataset_report = generate_dataset(DATASET, config, progress=True) if not DATASET.exists() else validate_dataset(DATASET)
dataset_report

## Modelli

Il baseline è il vincitore precedente. Il controllo mantiene la coordinata 8 globale e allarga la testa. Il candidato usa per la coordinata 8 la storia di `(x0, x8, u0, u1, u2, u3)`.

In [ ]:
EXPERIMENTS = {
    'receptor_composite': dict(architecture='conv_lstm_receptor_gru', global_head_dim=None),
    'receptor_capacity_control': dict(architecture='conv_lstm_receptor_gru', global_head_dim=283),
    'receptor_hcn_composite': dict(architecture='conv_lstm_receptor_hcn_gru', global_head_dim=None),
}
COMMON = dict(hidden_dim=128, layers=3, width_multiplier=2, receptor_hidden_dim=32, receptor_layers=1, hcn_hidden_dim=32, hcn_layers=1)
for settings in EXPERIMENTS.values():
    kwargs = {**COMMON, **{k: v for k, v in settings.items() if k != 'architecture'}}
    probe = build_model(settings['architecture'], len(STATE_NAMES)+len(INPUT_NAMES), len(STATE_NAMES), **kwargs)
    settings['parameters'] = sum(p.numel() for p in probe.parameters())
parameter_table = pd.DataFrame(EXPERIMENTS).T[['architecture', 'parameters']]
parameter_table

## Training controllato al 25% e 100%

Split, seed, optimizer, finestre e early stopping sono identici. Ogni epoca mostra tempo ed ETA.

In [ ]:
reports = []
for fraction in (0.25, 1.0):
    for name, settings in EXPERIMENTS.items():
        run_name = f'{name}_data_{int(100*fraction):03d}'
        print('\n' + '='*100); print(f'Training {run_name}: {settings["parameters"]:,} parameters')
        reports.append(train_model(
            DATASET, OUTPUT_DIR/'models', settings['architecture'], run_name=run_name, train_fraction=fraction,
            epochs=40, sequence_length=128, stride=32, batch_size=32, learning_rate=6e-4, dropout=.1,
            patience=8, minimum_epochs=18, device=DEVICE, seed=27182, use_amp=True, verbose=True,
            global_head_dim=settings.get('global_head_dim'), **COMMON,
        ))
print('All controlled runs completed.')

In [ ]:
comparison = pd.DataFrame([{
    'run': r['run_name'], 'model': r['run_name'].rsplit('_data_',1)[0], 'data_fraction': r['train_fraction'],
    'train_trajectories': r['train_trajectories'], 'parameters': r['parameters'], 'epochs': r['epochs_trained'],
    'validation_loss': r['best_validation_loss'], 'test_voltage_rmse_mV': r['test']['voltage_rmse_mv'],
    'test_normalized_rmse': r['test']['mean_normalized_rmse'],
} for r in reports]).sort_values(['data_fraction','validation_loss'])
comparison.to_csv(OUTPUT_DIR/'comparison.csv', index=False); comparison

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
for name, frame in comparison.groupby('model'):
    frame=frame.sort_values('train_trajectories')
    axes[0].plot(frame.train_trajectories,frame.test_voltage_rmse_mV,marker='o',label=name)
    axes[1].plot(frame.train_trajectories,frame.test_normalized_rmse,marker='o',label=name)
axes[0].set(title='Voltage',xlabel='training trajectories',ylabel='test RMSE (mV)')
axes[1].set(title='All states',xlabel='training trajectories',ylabel='mean normalized RMSE')
for axis in axes: axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'data_efficiency.png',dpi=160)

In [ ]:
full_reports=[r for r in reports if r['train_fraction']==1.0]
entity_errors=pd.DataFrame([{'model':r['run_name'].rsplit('_data_',1)[0],'entity':e,'normalized_rmse':v} for r in full_reports for e,v in r['test']['per_group_normalized_rmse'].items()])
entity_errors.to_csv(OUTPUT_DIR/'entity_errors.csv',index=False)
entity_errors.pivot(index='entity',columns='model',values='normalized_rmse').sort_values('receptor_hcn_composite',ascending=False)

In [ ]:
with h5py.File(DATASET,'r') as h5:
    truth=h5['test/states'][...]; future_inputs=h5['test/inputs'][...]
rollout_rows=[]; predictions={}; normalization=None
for report in full_reports:
    name=report['run_name'].rsplit('_data_',1)[0]
    checkpoint=torch.load(report['checkpoint'],map_location=DEVICE,weights_only=False)
    model=build_model(checkpoint['architecture'],len(STATE_NAMES)+len(INPUT_NAMES),len(STATE_NAMES),**checkpoint['model_kwargs']).to(DEVICE)
    model.load_state_dict(checkpoint['model_state']); normalization=Normalization.from_dict(checkpoint['normalization'])
    print(f'\nRollout {name} on all test trajectories...')
    prediction=rollout_batch(model,truth[:,0],future_inputs,normalization,DEVICE,progress=True); predictions[name]=prediction
    for horizon in (50,100,200,500):
        end=int(horizon/config.dt_ms)+1; error=prediction[:,:end]-truth[:,:end]
        pred_spikes=((prediction[:,1:end,0]>=config.membrane.spike_threshold_mv)&(prediction[:,:end-1,0]<config.membrane.spike_threshold_mv)).sum()
        true_spikes=((truth[:,1:end,0]>=config.membrane.spike_threshold_mv)&(truth[:,:end-1,0]<config.membrane.spike_threshold_mv)).sum()
        rollout_rows.append({'model':name,'horizon_ms':horizon,'voltage_rmse_mV':float(np.sqrt(np.mean(error[...,0]**2))),'mean_normalized_rmse':float(np.sqrt(np.mean((error/normalization.state_std)**2,axis=(0,1))).mean()),'teacher_spikes':int(true_spikes),'predicted_spikes':int(pred_spikes)})
rollout_table=pd.DataFrame(rollout_rows); rollout_table.to_csv(OUTPUT_DIR/'rollout_comparison.csv',index=False); rollout_table

In [ ]:
time_ms=np.arange(truth.shape[1])*config.dt_ms
fig,axes=plt.subplots(3,1,figsize=(15,9),sharex=True)
for axis,index,scale,label in zip(axes,(0,1,8),(1,1e3,1),('V (mV)','state 1 × 1000','state 8')):
    axis.plot(time_ms,truth[0,:,index]*scale,color='black',label='teacher',lw=1.2)
    for name,prediction in predictions.items(): axis.plot(time_ms,prediction[0,:,index]*scale,label=name,alpha=.8)
    axis.set_ylabel(label); axis.grid(alpha=.2); axis.legend()
axes[-1].set_xlabel('time (ms)'); fig.suptitle('Coordinate-8 composite rollout')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'rollout_example.png',dpi=160)

## Criterio di falsificazione

La separabilità della coordinata 8 è accettata soltanto se il candidato supera sia il composito precedente sia il suo controllo capacity-matched, migliora direttamente la coordinata 8 e non destabilizza rollout o coordinate globali.

In [ ]:
from shutil import copytree,make_archive,rmtree
import base64,os
from IPython.display import Javascript,display
parameter_table.to_csv(OUTPUT_DIR/'parameter_table.csv')
(OUTPUT_DIR/'experiment_definition.json').write_text(json.dumps({'fractions':[.25,1.0],'experiments':EXPERIMENTS,'common':COMMON,'seed':config.seed},indent=2),encoding='utf-8')
include_checkpoints=os.environ.get('HAY_DOWNLOAD_CHECKPOINTS','0')=='1'; archive_source=OUTPUT_DIR
staging=Path('/kaggle/working/hay_composite_03_download')
if not include_checkpoints:
    if staging.exists(): rmtree(staging)
    copytree(OUTPUT_DIR,staging,ignore=lambda path,names:{name for name in names if name.endswith('.pt')}); archive_source=staging
zip_path=Path(make_archive('/kaggle/working/hay_composite_experiment_03_complete','zip',root_dir=archive_source.parent,base_dir=archive_source.name))
encoded=base64.b64encode(zip_path.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{encoded}'),x=new Uint8Array(b.length);for(let i=0;i<b.length;i++)x[i]=b.charCodeAt(i);const o=URL.createObjectURL(new Blob([x],{{type:'application/zip'}})),a=document.createElement('a');a.href=o;a.download='{zip_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(o),60000);"""))
print('Download avviato:',zip_path,f'({zip_path.stat().st_size/2**20:.1f} MiB)','| checkpoint inclusi:',include_checkpoints)